# Türkiye EV Şarj İstasyonları - Analiz

Bu notebook, Türkiye genelindeki elektrikli araç şarj altyapısına ait bir veri setinin
keşifsel veri analizini (EDA) içermektedir.

**Veri seti:** 45.155 soket, 16.062 istasyon, 91 il, 182 marka
**Sütunlar:** Station_ID, Socket_ID, Is_Green_Energy, Service_Type, Brand,
Current_Type, Connector_Type, Power_kW, Latitude, Longitude, Province, District

**İçindekiler:**
1. Veri Yükleme ve Genel Bakış
2. Coğrafi Dağılım
3. Teknik Altyapı Profili (AC/DC, Konektör, Güç, Yeşil Enerji)
4. Marka/Operatör Analizi
5. Normalize Edilmiş İl Metrikleri
6. Cross-tab / Korelasyon Analizleri
7. Lorenz Eğrisi ve Gini Katsayısı
8. İl Segmentasyonu (K-Means)
9. Özet Rapor

## 1. Veri Yükleme ve Genel Bakış

In [ ]:
import pandas as pd 
import matplotlib.pyplot as plt

df = pd.read_csv("/kaggle/input/datasets/cokunkaanaki/trkiye-ev-arj-istasyonlar-data-set/ev_charging_stations_turkey.csv")

In [ ]:
stations = df.drop_duplicates(subset="Station_ID")

print("toplam soket sayısı",len(df))

print("toplam istasyon sayısı",df["Station_ID"].nunique())
print(df.head())
print(df.info())
print(df.isnull().sum())

## 2. Coğrafi Dağılım
En yoğun illeri ve genel coğrafi dağılımı inceliyoruz.

In [ ]:
top_provinces = df["Province"].value_counts().head(15)
print(top_provinces)

plt.figure(figsize=(9,6))
top_provinces.sort_values().plot(kind='barh', color='#2E7D32')
plt.title("illere göre soket sayısı (ilk 15)")
plt.xlabel("soket sayısı")
plt.tight_layout()
plt.savefig("top_provinces.png",dpi=150)
plt.show

In [ ]:
top_provinces_station = stations["Province"].value_counts().head(15)


## 3. Teknik Altyapı Profili
AC/DC oranı, konektör tipi, güç seviyeleri ve yeşil enerji kullanımı.

In [ ]:
current_counts = df["Current_Type"].value_counts()
print("current_counts")
print((current_counts / current_counts.sum()*100).round(1))

plt.figure(figsize=(6,6))
plt.pie(current_counts.values,labels=current_counts.index,autopct="%1.1f%%",
        colors= ["#1565C0", "#EF6C00"], startangle=90)
plt.title("AC vs DC dağılımı ")
plt.tight_layout()
plt.savefig("ac_vs_dc.png",dpi=150)
plt.show()

In [ ]:
connector_counts = df["Connector_Type"].value_counts()
print("connector_counts")

In [ ]:
print(df["Power_kW"].describe())

plt.figure(figsize=(9,5))
df["Power_kW"].value_counts().head(10).sort_values().plot(kind='barh',color="#1565C0")
plt.title("En Yaygın Güç Seviyeleri (KW)")
plt.xlabel("Soket Sayısı")
plt.tight_layout()
plt.savefig("Power.png",dpi=150)
plt.show()

In [ ]:
green_counts=df["Is_Green_Energy"].value_counts()
print((green_counts/green_counts.sum()*100).round(1))

top10_prov_names= df["Province"].value_counts().head(10).index
green_by_provice = (df[df["Province"].isin(top10_prov_names)]
                    .groupby("Province")["Is_Green_Energy"]
                    .mean()*100).sort_values()
plt.figure(figsize=(9,6))
green_by_provice.plot(kind='barh',color="#2E7D32")
plt.title("İlk 10 İlde Yeşil Enerji Oranı (%)")
plt.xlabel("Yeşil Enerji oranı ")
plt.tight_layout()
plt.savefig("green_by_province.png",dpi=150)
plt.show()
                    

## 4. Marka / Operatör Analizi

In [ ]:
top_brands = stations["Brand"].value_counts().head(12)
print(top_brands)

plt.figure(figsize=(9,6))
top_brands.sort_values().plot(kind='barh',color="#EF6C00")
plt.title("En Büyük 12 Marka (İstasyon Sayısı)")
plt.xlabel("İstasyon Sayısı")
plt.tight_layout()
plt.savefig("Brands.png",dpi=150)
plt.show()

In [ ]:
brand_counts=stations["Brand"].value_counts()
top10_share=brand_counts.head(10).sum()/ brand_counts.sum() * 100
print(f"İlk 10 marka payı: %{top10_share:.1f}")


In [ ]:
sockets_per_station = df.groupby("Station_ID").size()
print(sockets_per_station.describe())
 
plt.figure(figsize=(9, 5))
sockets_per_station.clip(upper=15).plot(kind="hist", bins=15, color="#2E7D32", edgecolor="white")
plt.title("İstasyon Başına Soket Sayısı Dağılımı (15+ kırpıldı)")
plt.xlabel("Soket Sayısı")
plt.ylabel("İstasyon Sayısı")
plt.tight_layout()
plt.savefig("sockets_per_station.png", dpi=150)
plt.show()
 

In [ ]:
plt.figure(figsize=(7, 9))
plt.scatter(df["Longitude"], df["Latitude"], s=3, alpha=0.3, color="#1565C0")
plt.title("Türkiye Geneli Şarj İstasyonu Dağılımı")
plt.xlabel("Boylam")
plt.ylabel("Enlem")
plt.gca().set_aspect("equal")
plt.tight_layout()
plt.savefig("map.png", dpi=150)
plt.show()
 

## 5. Normalize Edilmiş İl Metrikleri
Ham soket sayısı yerine, nüfusa oranlı (100k kişi başına) karşılaştırma.

In [ ]:
service_counts = df["Service_Type"].value_counts()
print((service_counts / service_counts.sum() * 100).round(1))
 

In [ ]:
# TÜİK 2023 nüfus tahminleri 

population = {
    "İSTANBUL" : 15907951,
    "ANKARA" : 5803482,
    "İZMİR" : 4479525,
    "BURSA": 3194720,
    "ANTALYA": 2688004,
    "KONYA": 2277017,
    "ADANA": 2270298,
    "ŞANLIURFA": 2170110,
    "GAZİANTEP": 2170013,
    "KOCAELİ": 2079072,
    "MERSİN": 1916432,
    "DİYARBAKIR": 1804880,
    "KAYSERİ": 1441523,
    "MANİSA": 1450616,
    "BALIKESİR": 1257590,
    "MUĞLA": 1048185,
}

In [ ]:

socket_counts = df["Province"].value_counts()
rows = []
for prov, pop in population.items():
    if prov in socket_counts.index:
        sockets = socket_counts[prov]
        rows.append({
            "Province": prov,
            "Sockets": sockets,
            "Population": pop,
            "Sockets_per_100k": sockets / pop * 100000
        })

normalized_df = pd.DataFrame(rows).sort_values("Sockets_per_100k", ascending=False)
print(normalized_df)

plt.figure(figsize=(9,6))
normalized_df.set_index("Province")["Sockets_per_100k"].sort_values().plot(kind='barh',color="#2E7D32")
plt.title("100.000 Kişi Başına Soket Sayısı ")
plt.xlabel("Soket/100k nufus")
plt.tight_layout()
plt.savefig("sockets_per_capita.png",dpi=150)
plt.show()


In [ ]:
top10_brands = stations["Brand"].value_counts().head(10).index
sub_brands= df[df["Brand"].isin(top10_brands)]

brand_curret_ct = pd.crosstab(sub_brands["Brand"],sub_brands["Current_Type"],normalize= "index")*100
print(brand_curret_ct.round(1))

brand_curret_ct.plot(kind='barh',stacked=True,figsize=(9,6),color=["#1565C0","#EF6C00"])
plt.title("İlk 10 Marka AC/DC Oranı (%)")
plt.ylabel("Yüzde(%)")
plt.legend(title="Akım Tipi")
plt.tight_layout()
plt.savefig("brand_ac_dc.png", dpi=150)
plt.show()


## 6. Cross-tab / Korelasyon Analizleri
Marka × AC-DC oranı, hizmet tipi × yeşil enerji, marka × ortalama güç ilişkileri.

In [ ]:
green_service_ct= pd.crosstab(df["Service_Type"],df["Is_Green_Energy"],normalize="index")*100
print(green_service_ct)
green_service_ct.plot(kind="bar", stacked=True, figsize=(7, 5), color=["#9E9E9E", "#2E7D32"])

plt.title("Hizmet Tipine Göre Yeşil Enerji Oranı (%)")
plt.ylabel("Yüzde(%)")
plt.xticks(rotation=0)
plt.legend(title="Yeşil Enerji", labels=["Hayır","Evet"])
plt.savefig("green_by_service.png",dpi=150)
plt.show()

In [ ]:
brand_avg_power = df[df["Brand"].isin(top10_brands)].groupby("Brand")["Power_kW"].mean().sort_values()
print(brand_avg_power.round(1))
brand_avg_power.plot(kind='barh',figsize=(9,6),color="#EF6C00")
plt.title("İlk 10 Markada Ortalama Güç (kW)")
plt.xlabel("Ortalama Güç (kW)")
plt.tight_layout()
plt.savefig("brand_avg_power.png", dpi=150)
plt.show()

In [ ]:
df_corr = df.copy()
df_corr["Is_DC"] = (df_corr["Current_Type"] == "DC").astype(int)
df_corr["Is_Green_Num"] = df_corr["Is_Green_Energy"].astype(int)
df_corr["Is_Public"] = (df_corr["Service_Type"] == "Public").astype(int)

correlation_matrix = df_corr[["Power_kW", "Is_DC", "Is_Green_Num", "Is_Public"]].corr()
print(correlation_matrix.round(2))

## 7. Lorenz Eğrisi ve Gini Katsayısı
Marka ve il bazında pazar yoğunlaşmasının ölçülmesi.

In [ ]:
# 13. LORENZ EĞRİSİ VE GINI KATSAYISI

def lorenz_curve(values):
    """Değer serisinden (küçükten büyüğe sıralı) kümülatif % döner."""
    sorted_vals = np.sort(values)
    cum_vals = np.cumsum(sorted_vals) / np.sum(sorted_vals)
    cum_vals = np.insert(cum_vals, 0, 0)  # (0,0) noktasını ekle
    cum_pop = np.linspace(0, 1, len(cum_vals))
    return cum_pop, cum_vals

def gini_coefficient(values):
    """Gini katsayısını hesaplar (0=eşit, 1=tam yoğunlaşma)."""
    sorted_vals = np.sort(values)
    n = len(sorted_vals)
    cum_vals = np.cumsum(sorted_vals)
    return (n + 1 - 2 * np.sum(cum_vals) / cum_vals[-1]) / n


import numpy as np

# --- A) Marka bazında yoğunlaşma (istasyon sayısı) ---
brand_station_counts = stations["Brand"].value_counts().values
pop_a, cum_a = lorenz_curve(brand_station_counts)
gini_brand = gini_coefficient(brand_station_counts)

# - B) İl bazında yoğunlaşma (soket sayısı) ---
province_socket_counts = df["Province"].value_counts().values
pop_b, cum_b = lorenz_curve(province_socket_counts)
gini_province = gini_coefficient(province_socket_counts)

print(f"Marka Gini katsayısı: {gini_brand:.3f}")
print(f"İl Gini katsayısı: {gini_province:.3f}")

# -- Grafik ---
plt.figure(figsize=(7, 7))
plt.plot(pop_a, cum_a, label=f"Marka (Gini={gini_brand:.2f})", color="#EF6C00", linewidth=2)
plt.plot(pop_b, cum_b, label=f"İl (Gini={gini_province:.2f})", color="#2E7D32", linewidth=2)
plt.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Tam Eşitlik")
plt.title("Lorenz Eğrisi: Marka ve İl Yoğunlaşması")
plt.xlabel("Kümülatif % (Marka / İl)")
plt.ylabel("Kümülatif % (İstasyon / Soket)")
plt.legend()
plt.tight_layout()
plt.savefig("lorenz_curve.png", dpi=150)
plt.show()

## 8. İl Segmentasyonu (K-Means Kümeleme)
İlleri DC oranı, yeşil enerji oranı, hizmet tipi ve güç profiline göre gruplama.

In [ ]:
# 14. İL SEGMENTASYONU (K-MEANS KÜMELEME)
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# En az 30 soketi olan illeri al (çok küçük illerde   oranlar anlamsız dalgalanır)
province_socket_counts = df["Province"].value_counts()
valid_provinces = province_socket_counts[province_socket_counts >= 30].index
sub = df[df["Province"].isin(valid_provinces)].copy()

# İl bazında özellik tablosu 
province_features = sub.groupby("Province").agg(
    Total_Sockets=("Station_ID", "count"),
    DC_Ratio=("Current_Type", lambda x: (x == "DC").mean() * 100),
    Green_Ratio=("Is_Green_Energy", "mean"),
    Public_Ratio=("Service_Type", lambda x: (x == "Public").mean() * 100),
    Avg_Power=("Power_kW", "mean"),
)
province_features["Green_Ratio"] = province_features["Green_Ratio"] * 100

print(province_features.round(1))

# Özellikleri ölçeklendir 
features_for_clustering = ["DC_Ratio", "Green_Ratio", "Public_Ratio", "Avg_Power"]
X = province_features[features_for_clustering]
X_scaled = StandardScaler().fit_transform(X)

# K-Means uygula
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
province_features["Cluster"] = kmeans.fit_predict(X_scaled)

print(province_features.sort_values("Cluster"))

# Her kümenin ortalama profili
cluster_profile = province_features.groupby("Cluster")[features_for_clustering + ["Total_Sockets"]].mean()
print(cluster_profile.round(1))

## 9. Özet Rapor

In [ ]:
# 15. ÖZET RAPOR

print("=" * 60)
print("TÜRKİYE EV ŞARJ İSTASYONLARI - ÖZET RAPOR")
print("=" * 60)

print("\n--- GENEL ---")
print(f"Toplam soket sayısı: {len(df):,}")
print(f"Toplam istasyon sayısı: {df['Station_ID'].nunique():,}")
print(f"İl sayısı: {df['Province'].nunique()}")
print(f"Marka sayısı: {df['Brand'].nunique()}")

print("\n--- TEKNİK PROFİL ---")
dc_ratio = (df["Current_Type"] == "DC").mean() * 100
green_ratio = df["Is_Green_Energy"].mean() * 100
public_ratio = (df["Service_Type"] == "Public").mean() * 100
print(f"DC (hızlı şarj) oranı: %{dc_ratio:.1f}")
print(f"Yeşil enerji oranı: %{green_ratio:.1f}")
print(f"Public (kamuya açık) oranı: %{public_ratio:.1f}")
print(f"Ortalama güç: {df['Power_kW'].mean():.1f} kW")
print(f"Medyan güç: {df['Power_kW'].median():.1f} kW")

print("\n--- COĞRAFİ YOĞUNLAŞMA ---")
top5_share = df["Province"].value_counts().head(5).sum() / len(df) * 100
print(f"İlk 5 il, toplam soketlerin %{top5_share:.1f}'ini barındırıyor")
print(f"İl bazında Gini katsayısı: {gini_province:.3f}")

print("\n--- PAZAR YAPISI ---")
top10_brand_share = stations["Brand"].value_counts().head(10).sum() / len(stations) * 100
print(f"İlk 10 marka, istasyonların %{top10_brand_share:.1f}'ini kontrol ediyor")
print(f"Marka bazında Gini katsayısı: {gini_brand:.3f}")

print("\n--- KİŞİ BAŞINA EN İYİ 3 İL (Soket/100k Nüfus) ---")
print(normalized_df.head(3)[["Province", "Sockets_per_100k"]].to_string(index=False))

print("\n--- İL SEGMENTASYONU (Küme Profilleri) ---")
print(cluster_profile.round(1))

print("\n--- KORELASYON (Power_kW ile diğer değişkenler) ---")
print(correlation_matrix["Power_kW"].round(2))

print("\n" + "=" * 60)